# Protocol sensitivity on an open-data CPU surrogate

This notebook contains the numerical analysis for the local reproduction of
[*Quantum Kernel Advantage over Classical Collapse in Medical Foundation Model
Embeddings*](http://arxiv.org/abs/2604.24597).

It evaluates:

- the imported QSVM path;
- linear and tuned-RBF classical baselines;
- a separate MerLin photonic fidelity-kernel adaptation;
- the sensitivity of the results to preprocessing leakage and trace
  normalization.

The original controlled-access [MIMIC-CXR insurance-classification experiment](https://huggingface.co/datasets/MITCriticalData/qml-mimic-cxr-embeddings)
was not run. The local experiment uses raw [PneumoniaMNIST](https://www.nature.com/articles/s41597-022-01721-8) pixels and must be
interpreted as a surrogate and implementation audit.

This notebook reads committed CSV files only. It does not recompute kernels,
train classifiers, or rerun experiments.

## 1. Reference claim and local questions

The paper uses frozen medical foundation-model embeddings compressed to `q`
PCA features.

Its primary comparisons report:

- 18/18 QSVM wins against an untuned linear SVM;
- 7/7 wins against a validation-tuned RBF SVM;
- minority-class F1 equal to zero for the linear SVM on most embedding seeds.

The local experiment does not directly test these claims. It asks:

1. How sensitive are the surrogate results to fitting `MinMaxScaler` on
   training plus held-out data rather than training data only?
2. How sensitive is the QSVM to scaling only the square training Gram matrix
   rather than scaling the associated cross-kernel with the same training
   trace?
3. How do the QSVM, classical baselines, and MerLin adaptation compare on
   paired local seeds?
4. How often does each model win, tie, or lose on the same subset and split?

## 2. Local setup

| Item | Local experiment |
|---|---|
| Reference reproduction | Not run |
| Dataset | PneumoniaMNIST training split |
| Representation | Raw flattened 28×28 pixels |
| Samples per seed | 500 |
| Split | 400 train / 50 validation / 50 test |
| Minority class | `normal` |
| Data/split seeds | 0–9 |
| PCA dimensions | `q=4`, `q=6` |
| QSVM | Imported BSP-derived path, `C=1` |
| Linear baseline | Linear SVM, `C=1` |
| RBF baseline | `C` selected on validation minority F1 |
| MerLin | Photonic fidelity kernel, `C=1`, circuit seed 0 |
| Primary metric | Test minority-class F1 |

The four protocol identifiers are:

| Protocol ID | MinMax fitting | QSVM trace | MerLin normalization |
|---|---|---|---|
| `legacy_leak__legacy_trace` | train + held-out | square only | unnormalized |
| `train_only__legacy_trace` | train only | square only | unnormalized |
| `legacy_leak__train_trace` | train + held-out | train trace | train trace |
| `train_only__train_trace` | train only | train trace | train trace |

For MerLin, `legacy_trace` is only a shared launcher label. MerLin is absent
from the imported repository and has no upstream normalization behavior.

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


paper_dir = Path.cwd()

if not (paper_dir / "results").exists():
    candidate = paper_dir / "papers" / "qsvm_medimage"
    if candidate.exists():
        paper_dir = candidate

results_dir = (
    paper_dir
    / "results"
    / "protocol_matrix_n500_q4_q6"
)

summary_path = results_dir / "protocol_summary.csv"
per_seed_path = results_dir / "protocol_results_per_seed.csv"

summary = pd.read_csv(summary_path)
per_seed = pd.read_csv(per_seed_path)


required_summary_columns = {
    "protocol_id",
    "preprocessing_protocol",
    "trace_protocol",
    "kernel_normalization",
    "q",
    "model",
    "baseline",
    "mean_f1",
    "std_f1",
    "baseline_mean_f1",
    "baseline_std_f1",
    "delta_f1",
    "wins",
    "ties",
    "losses",
    "seeds",
}

required_per_seed_columns = {
    "protocol_id",
    "preprocessing_protocol",
    "trace_protocol",
    "kernel_normalization",
    "seed",
    "q",
    "model",
    "model_f1",
    "baseline",
    "baseline_f1",
    "delta_f1",
}

missing_summary = required_summary_columns - set(summary.columns)
missing_per_seed = required_per_seed_columns - set(per_seed.columns)

if missing_summary:
    raise ValueError(
        f"Missing protocol-summary columns: {sorted(missing_summary)}"
    )

if missing_per_seed:
    raise ValueError(
        f"Missing per-seed columns: {sorted(missing_per_seed)}"
    )

if len(summary) != 32:
    raise ValueError(
        f"Expected 32 aggregate rows, found {len(summary)}"
    )

if not (summary["wins"] + summary["ties"] + summary["losses"]).eq(
    summary["seeds"]
).all():
    raise ValueError("At least one W/T/L count does not match its seed count")

display(
    Markdown(
        f"Loaded **{len(summary)} aggregate rows** and "
        f"**{len(per_seed)} paired model–baseline rows**."
    )
)

Loaded **32 aggregate rows** and **320 paired model–baseline rows**.

In [19]:
protocol_order = [
    "legacy_leak__legacy_trace",
    "train_only__legacy_trace",
    "legacy_leak__train_trace",
    "train_only__train_trace",
]

preprocessing_labels = {
    "legacy_train_plus_heldout": "train + held-out",
    "train_only": "train only",
}

trace_labels = {
    "legacy_square_only": "square only",
    "train_trace": "train trace",
}

merlin_normalization_labels = {
    "none": "unnormalized",
    "train_trace": "train trace",
}


def format_mean_std(mean_value, std_value):
    """Format one aggregate result with two decimal places."""
    return f"{float(mean_value):.2f} ± {float(std_value):.2f}"


def format_delta(value):
    """Format a signed delta while avoiding '+0.00' and '-0.00'."""
    rounded = round(float(value), 2)

    if rounded == 0:
        return "0.00"

    return f"{rounded:+.2f}"


def format_wtl(wins, ties, losses):
    """Format seed-level Wins / Ties / Losses."""
    return f"{int(wins)}/{int(ties)}/{int(losses)}"


def dataframe_to_markdown(frame):
    """Create a Markdown table without requiring the `tabulate` package."""
    columns = [str(column) for column in frame.columns]

    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]

    for row in frame.astype(str).itertuples(index=False, name=None):
        lines.append("| " + " | ".join(row) + " |")

    return "\n".join(lines)


def show_table(frame):
    display(Markdown(dataframe_to_markdown(frame)))

## 3. QSVM protocol sensitivity

The table reports mean test minority-class F1 ± sample standard deviation over
ten paired data/split seeds.

Definitions:

- `Q-linear = QSVM F1 - linear SVM F1`;
- `Q-RBF = QSVM F1 - tuned RBF SVM F1`;
- positive deltas favor the QSVM;
- W/T/L means Wins / Ties / Losses, counted seed by seed from the QSVM
  perspective.

Deltas are displayed with two decimal places. W/T/L is calculated from the
unrounded per-seed values, so a displayed delta of `0.00` does not necessarily
mean that all seeds are ties.

In [20]:
def select_summary_rows(model, baseline):
    return summary[
        (summary["model"] == model)
        & (summary["baseline"] == baseline)
    ].copy()


qsvm_linear = select_summary_rows(
    model="qsvm",
    baseline="linear_c1",
).rename(
    columns={
        "baseline_mean_f1": "linear_mean_f1",
        "baseline_std_f1": "linear_std_f1",
        "delta_f1": "linear_delta_f1",
        "wins": "linear_wins",
        "ties": "linear_ties",
        "losses": "linear_losses",
    }
)

qsvm_rbf = select_summary_rows(
    model="qsvm",
    baseline="rbf_tuned",
).rename(
    columns={
        "baseline_mean_f1": "rbf_mean_f1",
        "baseline_std_f1": "rbf_std_f1",
        "delta_f1": "rbf_delta_f1",
        "wins": "rbf_wins",
        "ties": "rbf_ties",
        "losses": "rbf_losses",
    }
)

qsvm_keys = [
    "protocol_id",
    "preprocessing_protocol",
    "trace_protocol",
    "q",
]

qsvm_table = qsvm_linear[
    qsvm_keys
    + [
        "mean_f1",
        "std_f1",
        "linear_mean_f1",
        "linear_std_f1",
        "linear_delta_f1",
        "linear_wins",
        "linear_ties",
        "linear_losses",
    ]
].merge(
    qsvm_rbf[
        qsvm_keys
        + [
            "rbf_mean_f1",
            "rbf_std_f1",
            "rbf_delta_f1",
            "rbf_wins",
            "rbf_ties",
            "rbf_losses",
        ]
    ],
    on=qsvm_keys,
    validate="one_to_one",
)

qsvm_table["protocol_order"] = pd.Categorical(
    qsvm_table["protocol_id"],
    categories=protocol_order,
    ordered=True,
)

qsvm_table = qsvm_table.sort_values(
    ["protocol_order", "q"]
).reset_index(drop=True)

qsvm_display = pd.DataFrame(
    {
        "Preprocessing": qsvm_table[
            "preprocessing_protocol"
        ].map(preprocessing_labels),
        "QSVM trace": qsvm_table[
            "trace_protocol"
        ].map(trace_labels),
        "q": qsvm_table["q"].astype(int),
        "QSVM F1": [
            format_mean_std(mean, std)
            for mean, std in zip(
                qsvm_table["mean_f1"],
                qsvm_table["std_f1"],
            )
        ],
        "Linear F1": [
            format_mean_std(mean, std)
            for mean, std in zip(
                qsvm_table["linear_mean_f1"],
                qsvm_table["linear_std_f1"],
            )
        ],
        "Q-linear": qsvm_table[
            "linear_delta_f1"
        ].map(format_delta),
        "vs linear W/T/L": [
            format_wtl(wins, ties, losses)
            for wins, ties, losses in zip(
                qsvm_table["linear_wins"],
                qsvm_table["linear_ties"],
                qsvm_table["linear_losses"],
            )
        ],
        "Tuned RBF F1": [
            format_mean_std(mean, std)
            for mean, std in zip(
                qsvm_table["rbf_mean_f1"],
                qsvm_table["rbf_std_f1"],
            )
        ],
        "Q-RBF": qsvm_table[
            "rbf_delta_f1"
        ].map(format_delta),
        "vs RBF W/T/L": [
            format_wtl(wins, ties, losses)
            for wins, ties, losses in zip(
                qsvm_table["rbf_wins"],
                qsvm_table["rbf_ties"],
                qsvm_table["rbf_losses"],
            )
        ],
    }
)

show_table(qsvm_display)

| Preprocessing | QSVM trace | q | QSVM F1 | Linear F1 | Q-linear | vs linear W/T/L | Tuned RBF F1 | Q-RBF | vs RBF W/T/L |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| train + held-out | square only | 4 | 0.81 ± 0.06 | 0.76 ± 0.08 | +0.05 | 7/0/3 | 0.76 ± 0.12 | +0.05 | 6/1/3 |
| train + held-out | square only | 6 | 0.82 ± 0.08 | 0.79 ± 0.09 | +0.04 | 7/0/3 | 0.82 ± 0.09 | 0.00 | 4/1/5 |
| train only | square only | 4 | 0.81 ± 0.06 | 0.76 ± 0.08 | +0.06 | 8/0/2 | 0.76 ± 0.12 | +0.05 | 6/1/3 |
| train only | square only | 6 | 0.82 ± 0.08 | 0.79 ± 0.09 | +0.04 | 7/0/3 | 0.81 ± 0.09 | +0.01 | 4/1/5 |
| train + held-out | train trace | 4 | 0.00 ± 0.00 | 0.76 ± 0.08 | -0.76 | 0/0/10 | 0.76 ± 0.12 | -0.76 | 0/0/10 |
| train + held-out | train trace | 6 | 0.00 ± 0.00 | 0.79 ± 0.09 | -0.79 | 0/0/10 | 0.82 ± 0.09 | -0.82 | 0/0/10 |
| train only | train trace | 4 | 0.00 ± 0.00 | 0.76 ± 0.08 | -0.76 | 0/0/10 | 0.76 ± 0.12 | -0.76 | 0/0/10 |
| train only | train trace | 6 | 0.00 ± 0.00 | 0.79 ± 0.09 | -0.79 | 0/0/10 | 0.81 ± 0.09 | -0.81 | 0/0/10 |

### QSVM interpretation

The MinMax-fitting choice has little effect on this surrogate.

The trace protocol changes the result decisively:

- with square-only scaling, QSVM minority F1 is non-zero;
- with consistent train-trace scaling, QSVM minority F1 is zero on every seed
  at both `q=4` and `q=6`.

The square training kernel is trace-normalized in both QSVM trace variants.
The controlled difference is therefore the scale of the validation/test
cross-kernel.

The favorable held-out scores of the preserved imported path depend on this
scale mismatch in the local surrogate.

The paper's central linear-collapse-avoidance mechanism is not reproduced:
the local linear baseline itself has non-zero minority F1.

## 4. MerLin photonic adaptation

MerLin is not part of the paper or the imported repository.

The local adaptation uses:

- the same 500-sample subset;
- the same train/validation/test split;
- the same PCA dimension;
- a fixed photonic circuit seed equal to 0;
- an exact CPU fidelity kernel;
- an SVM with fixed `C=1`.

The table compares MerLin with:

- the local QSVM;
- the linear SVM;
- the tuned RBF SVM.

Definitions:

- `M-QSVM = MerLin F1 - QSVM F1`;
- `M-linear = MerLin F1 - linear SVM F1`;
- `M-RBF = MerLin F1 - tuned RBF SVM F1`;
- positive deltas favor MerLin;
- W/T/L is counted seed by seed from the MerLin perspective.

The MerLin-versus-QSVM deltas and W/T/L counts are derived from the committed
per-seed results. They are not entered manually.

## 4. MerLin photonic adaptation

For every q–seed pair, MerLin receives the same N=500 subset, 400/50/50 split, and PCA dimension as the three Claim 1 models. The data seed varies from 0 to 9, while the MerLin circuit seed remains fixed at 0.

| Property | Matched setting |
|---|---|
| Samples | N=500 |
| Dimensions | PCA q=4 and q=6 |
| Data/split seeds | 0–9 |
| Classifier | Precomputed-kernel SVM, `C=1` |
| Photonic resources at q=4 | 5 modes, 3 photons, input `[1, 0, 1, 0, 1]` |
| Photonic resources at q=6 | 7 modes, 4 photons, input `[1, 0, 1, 0, 1, 0, 1]` |
| Circuit seed | 0, fixed |

MerLin is absent from the paper and imported upstream repository. This removes sample-and-split differences from the local comparison, but it does not make MerLin equivalent or resource-matched to the BSP QSVM. The evaluated variants are the MerLin unnormalized variant and the MerLin train-trace variant.

In [21]:
merlin_linear = select_summary_rows(
    model="merlin_fidelity",
    baseline="linear_c1",
).rename(
    columns={
        "baseline_mean_f1": "linear_mean_f1",
        "baseline_std_f1": "linear_std_f1",
        "delta_f1": "linear_delta_f1",
        "wins": "linear_wins",
        "ties": "linear_ties",
        "losses": "linear_losses",
    }
)

merlin_rbf = select_summary_rows(
    model="merlin_fidelity",
    baseline="rbf_tuned",
).rename(
    columns={
        "baseline_mean_f1": "rbf_mean_f1",
        "baseline_std_f1": "rbf_std_f1",
        "delta_f1": "rbf_delta_f1",
        "wins": "rbf_wins",
        "ties": "rbf_ties",
        "losses": "rbf_losses",
    }
)

merlin_keys = [
    "protocol_id",
    "preprocessing_protocol",
    "kernel_normalization",
    "q",
]

merlin_table = merlin_linear[
    merlin_keys
    + [
        "mean_f1",
        "std_f1",
        "linear_delta_f1",
        "linear_wins",
        "linear_ties",
        "linear_losses",
    ]
].merge(
    merlin_rbf[
        merlin_keys
        + [
            "rbf_delta_f1",
            "rbf_wins",
            "rbf_ties",
            "rbf_losses",
        ]
    ],
    on=merlin_keys,
    validate="one_to_one",
)


# Each model score appears twice in the per-seed file, once per classical
# baseline. Keep one unique score per protocol, seed, q, and model.
model_scores = per_seed[
    [
        "protocol_id",
        "preprocessing_protocol",
        "seed",
        "q",
        "model",
        "model_f1",
    ]
].drop_duplicates()

merlin_scores = model_scores[
    model_scores["model"] == "merlin_fidelity"
].rename(
    columns={"model_f1": "merlin_f1"}
)

qsvm_scores = model_scores[
    model_scores["model"] == "qsvm"
].rename(
    columns={"model_f1": "qsvm_f1"}
)

merlin_vs_qsvm = merlin_scores.merge(
    qsvm_scores[
        [
            "protocol_id",
            "seed",
            "q",
            "qsvm_f1",
        ]
    ],
    on=[
        "protocol_id",
        "seed",
        "q",
    ],
    validate="one_to_one",
)

merlin_vs_qsvm["m_qsvm_delta"] = (
    merlin_vs_qsvm["merlin_f1"]
    - merlin_vs_qsvm["qsvm_f1"]
)

tie_tolerance = 1e-12

merlin_vs_qsvm["is_tie"] = np.isclose(
    merlin_vs_qsvm["m_qsvm_delta"],
    0.0,
    atol=tie_tolerance,
    rtol=0.0,
)

merlin_vs_qsvm["is_win"] = (
    ~merlin_vs_qsvm["is_tie"]
    & (merlin_vs_qsvm["m_qsvm_delta"] > 0)
)

merlin_vs_qsvm["is_loss"] = (
    ~merlin_vs_qsvm["is_tie"]
    & (merlin_vs_qsvm["m_qsvm_delta"] < 0)
)

merlin_qsvm_summary = (
    merlin_vs_qsvm
    .groupby(
        [
            "protocol_id",
            "q",
        ],
        as_index=False,
    )
    .agg(
        m_qsvm_delta=("m_qsvm_delta", "mean"),
        m_qsvm_wins=("is_win", "sum"),
        m_qsvm_ties=("is_tie", "sum"),
        m_qsvm_losses=("is_loss", "sum"),
    )
)

merlin_table = merlin_table.merge(
    merlin_qsvm_summary,
    on=[
        "protocol_id",
        "q",
    ],
    validate="one_to_one",
)

merlin_table["protocol_order"] = pd.Categorical(
    merlin_table["protocol_id"],
    categories=protocol_order,
    ordered=True,
)

merlin_table = merlin_table.sort_values(
    ["protocol_order", "q"]
).reset_index(drop=True)

merlin_display = pd.DataFrame(
    {
        "Preprocessing": merlin_table[
            "preprocessing_protocol"
        ].map(preprocessing_labels),
        "MerLin normalization": merlin_table[
            "kernel_normalization"
        ].map(merlin_normalization_labels),
        "q": merlin_table["q"].astype(int),
        "MerLin F1": [
            format_mean_std(mean, std)
            for mean, std in zip(
                merlin_table["mean_f1"],
                merlin_table["std_f1"],
            )
        ],
        "M-QSVM": merlin_table[
            "m_qsvm_delta"
        ].map(format_delta),
        "vs QSVM W/T/L": [
            format_wtl(wins, ties, losses)
            for wins, ties, losses in zip(
                merlin_table["m_qsvm_wins"],
                merlin_table["m_qsvm_ties"],
                merlin_table["m_qsvm_losses"],
            )
        ],
        "M-linear": merlin_table[
            "linear_delta_f1"
        ].map(format_delta),
        "vs linear W/T/L": [
            format_wtl(wins, ties, losses)
            for wins, ties, losses in zip(
                merlin_table["linear_wins"],
                merlin_table["linear_ties"],
                merlin_table["linear_losses"],
            )
        ],
        "M-RBF": merlin_table[
            "rbf_delta_f1"
        ].map(format_delta),
        "vs RBF W/T/L": [
            format_wtl(wins, ties, losses)
            for wins, ties, losses in zip(
                merlin_table["rbf_wins"],
                merlin_table["rbf_ties"],
                merlin_table["rbf_losses"],
            )
        ],
    }
)

show_table(merlin_display)

| Preprocessing | MerLin normalization | q | MerLin F1 | M-QSVM | vs QSVM W/T/L | M-linear | vs linear W/T/L | M-RBF | vs RBF W/T/L |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| train + held-out | unnormalized | 4 | 0.76 ± 0.13 | -0.05 | 4/0/6 | 0.00 | 6/0/4 | 0.00 | 4/2/4 |
| train + held-out | unnormalized | 6 | 0.77 ± 0.10 | -0.05 | 3/0/7 | -0.02 | 2/3/5 | -0.05 | 1/1/8 |
| train only | unnormalized | 4 | 0.76 ± 0.13 | -0.06 | 4/0/6 | 0.00 | 6/0/4 | 0.00 | 4/2/4 |
| train only | unnormalized | 6 | 0.77 ± 0.10 | -0.05 | 3/0/7 | -0.02 | 2/3/5 | -0.04 | 1/1/8 |
| train + held-out | train trace | 4 | 0.00 ± 0.00 | 0.00 | 0/10/0 | -0.76 | 0/0/10 | -0.76 | 0/0/10 |
| train + held-out | train trace | 6 | 0.00 ± 0.00 | 0.00 | 0/10/0 | -0.79 | 0/0/10 | -0.82 | 0/0/10 |
| train only | train trace | 4 | 0.00 ± 0.00 | 0.00 | 0/10/0 | -0.76 | 0/0/10 | -0.76 | 0/0/10 |
| train only | train trace | 6 | 0.00 ± 0.00 | 0.00 | 0/10/0 | -0.79 | 0/0/10 | -0.81 | 0/0/10 |

### MerLin interpretation

Without trace normalization, MerLin is in the same broad minority-F1 range as
the local QSVM and classical baselines:

- it is close to the classical baselines at `q=4`;
- it is below the QSVM on average at both dimensions;
- it is below the classical baselines at `q=6`.

With train-trace normalization and fixed `C=1`, both MerLin and the QSVM have
zero minority F1 on every seed.

For the QSVM audit, the trace comparison isolates the held-out cross-kernel
scale because the square training kernel is normalized in both variants.

For MerLin, switching from `unnormalized` to `train trace` changes the scale of
both the training kernel and its cross-kernels. The result therefore does not
indicate an upstream-style MerLin bug.

It shows that the current MerLin configuration is sensitive to the joint choice
of kernel normalization and fixed SVM regularization.

Tuning `C`, varying the photonic circuit seed, and exploring different feature
maps or photon/mode resources remain future work.

In [24]:
# Keep one score per model, protocol and q.
# In protocol_summary.csv, the same model F1 appears once for the linear
# comparison and once for the RBF comparison.
model_results = (
    summary[
        [
            "protocol_id",
            "preprocessing_protocol",
            "trace_protocol",
            "kernel_normalization",
            "q",
            "model",
            "mean_f1",
        ]
    ]
    .drop_duplicates()
    .copy()
)

# QSVM stores its normalization in `trace_protocol`, while MerLin stores it in
# `kernel_normalization`. Merge both into one descriptive column.
model_results["normalization"] = (
    model_results["trace_protocol"]
    .fillna(model_results["kernel_normalization"])
)

# Put the leaky and train-only results side by side.
preprocessing_comparison = (
    model_results
    .pivot(
        index=[
            "model",
            "q",
            "normalization",
        ],
        columns="preprocessing_protocol",
        values="mean_f1",
    )
    .reset_index()
)

preprocessing_comparison["difference_percentage_points"] = (
    100
    * (
        preprocessing_comparison["legacy_train_plus_heldout"]
        - preprocessing_comparison["train_only"]
    ).abs()
)

preprocessing_display = preprocessing_comparison.rename(
    columns={
        "model": "Model",
        "q": "q",
        "normalization": "Normalization",
        "legacy_train_plus_heldout": "Train + held-out F1",
        "train_only": "Train-only F1",
        "difference_percentage_points": "Absolute difference (percentage points)",
    }
)

maximum_effect = preprocessing_comparison[
    "difference_percentage_points"
].max()

qsvm_all_zero = np.isclose(
    model_results.loc[
        (model_results["model"] == "qsvm")
        & (model_results["normalization"] == "train_trace"),
        "mean_f1",
    ],
    0.0,
    atol=1e-12,
).all()

merlin_all_zero = np.isclose(
    model_results.loc[
        (model_results["model"] == "merlin_fidelity")
        & (model_results["normalization"] == "train_trace"),
        "mean_f1",
    ],
    0.0,
    atol=1e-12,
).all()

print(
    "Largest effect of changing only the MinMax-fitting protocol: "
    f"{maximum_effect:.2f} percentage points"
)

print(
    "QSVM train-trace F1 is zero in every aggregated configuration:",
    bool(qsvm_all_zero),
)

print(
    "MerLin train-trace F1 is zero in every aggregated configuration:",
    bool(merlin_all_zero),
)

Largest effect of changing only the MinMax-fitting protocol: 0.48 percentage points
QSVM train-trace F1 is zero in every aggregated configuration: True
MerLin train-trace F1 is zero in every aggregated configuration: True


## 5. Main conclusions

1. **The MinMax leakage has little empirical effect on this surrogate.**

   The largest aggregate minority-F1 change between matching leaky and
   train-only protocols is below `0.005`.

   It remains an invalid evaluation protocol, but it does not explain the
   favorable imported-path result here.

2. **The QSVM trace mismatch is decisive in this local experiment.**

   With consistent train-trace scaling, QSVM minority F1 is zero on every seed
   at `q=4` and `q=6`.

3. **The paper's central collapse-avoidance mechanism is not reproduced.**

   The local linear SVM has non-zero minority F1, unlike the collapse reported
   for the paper's frozen MIMIC-CXR embeddings.

4. **MerLin follows the same qualitative fixed-`C` normalization
   sensitivity.**

   This is a result of the local photonic adaptation and does not represent an
   upstream MerLin behavior.

5. **The local experiment neither reproduces nor refutes the paper.**

   It uses a different dataset, target, representation, sample count, seed
   definition, and reduced `q` range.

## 6. Limitations

- The controlled-access MIMIC-CXR reproduction was not run.
- Pneumonia classification differs from insurance classification.
- Raw pixels differ from frozen foundation-model embeddings.
- Only `q=4` and `q=6` were evaluated.
- Each test split contains only 50 samples.
- Local seeds control subsets and splits, not embedding generation.
- No local paired significance test was run.
- The local linear baseline does not exhibit the collapse central to the
  paper.
- MerLin is not a faithful implementation or resource match of the BSP qubit
  circuit.
- The MerLin feature map, circuit seed, kernel scale, and SVM `C` were not
  optimized.
- These local findings do not establish whether the paper's original
  MIMIC-CXR results are valid.

## 7. Result artifacts

The notebook reads:

- [`protocol_summary.csv`](results/protocol_matrix_n500_q4_q6/protocol_summary.csv);
- [`protocol_results_per_seed.csv`](results/protocol_matrix_n500_q4_q6/protocol_results_per_seed.csv).

Additional provenance:

- [`protocol_summary.md`](results/protocol_matrix_n500_q4_q6/protocol_summary.md);
- [`RUN.md`](results/protocol_matrix_n500_q4_q6/RUN.md);
- [`q4_q6_n500_per_seed.csv`](results/q4_q6_n500_per_seed.csv), retained for
  the preserved imported-path result.

For implementation details and exact upstream-code pointers, see
[`README.md`](README.md) and [`AUDIT.md`](AUDIT.md).